위 코드 실행 후 Kernel > Restart Kernel

In [1]:
import torch
import torchvision
import torchaudio
import transformers
import datasets
import peft
import vllm

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("vllm:", vllm.__version__)

torch: 2.8.0+cu128
torchvision: 0.23.0+cu128
torchaudio: 2.8.0+cu128
transformers: 4.57.6
datasets: 3.6.0
peft: 0.19.1
vllm: 0.10.2


In [2]:
from pathlib import Path
import ast
import re

import pandas as pd
import vllm
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from rouge import Rouge
from mecab import MeCab

INFO 05-30 08:29:27 [__init__.py:216] Automatically detected platform cuda.


In [3]:
llm = LLM(model="/workspace/models/Qwen3-4B")

INFO 05-30 08:29:49 [utils.py:328] non-default args: {'disable_log_stats': True, 'model': '/workspace/models/Qwen3-4B'}
INFO 05-30 08:30:01 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-30 08:30:01 [__init__.py:1815] Using max model len 40960
INFO 05-30 08:30:03 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:04 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:04 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='/workspace/models/Qwen3-4B', speculative_config=None, tokenizer='/workspace/models/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoni

[W530 08:30:07.423786721 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:08 [gpu_model_runner.py:2338] Starting to load model /workspace/models/Qwen3-4B...
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:08 [gpu_model_runner.py:2370] Loading model from scratch...
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:08 [cuda.py:362] Using Flash Attention backend on V1 engine.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore_DP0 pid=6437) INFO 05-30 08:30:20 [default_loader.py:268] Loading weights took 11.39 seconds
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:21 [gpu_model_runner.py:2392] Model loading took 7.5552 GiB and 11.943944 seconds
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:27 [backends.py:539] Using cache directory: /root/.cache/vllm/torch_compile_cache/58b69f4896/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:27 [backends.py:550] Dynamo bytecode transform time: 5.83 s
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:32 [backends.py:194] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=6437) INFO 05-30 08:30:57 [backends.py:215] Compiling a graph for dynamic shape takes 28.80 s
(EngineCore_DP0 pid=6437) INFO 05-30 08:31:01 [monitor.py:34] torch.compile takes 34.63 s in total
(EngineCore_DP0 pid=6437) INFO 05-30 08:31:03 [gpu_worker.py:298] Available KV cache memory: 62.73 GiB
(EngineCore_DP0 pid=6437) INFO 05-30 08:31:03 [kv_cache_util

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 20.87it/s]


(EngineCore_DP0 pid=6437) INFO 05-30 08:31:08 [gpu_model_runner.py:3118] Graph capturing finished in 4 secs, took 0.55 GiB
(EngineCore_DP0 pid=6437) INFO 05-30 08:31:08 [gpu_worker.py:391] Free memory on device (78.35/79.25 GiB) on startup. Desired GPU memory utilization is (0.9, 71.32 GiB). Actual usage is 7.56 GiB for weight, 1.43 GiB for peak activation, -0.39 GiB for non-torch memory, and 0.55 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=66616480563` to fit into requested memory, or `--kv-cache-memory=74158678016` to fully utilize gpu memory. Current kv cache memory in use is 67360969523 bytes.
(EngineCore_DP0 pid=6437) INFO 05-30 08:31:08 [core.py:218] init engine (profile, create kv cache, warmup model) took 46.69 seconds
INFO 05-30 08:31:09 [llm.py:295] Supported_tasks: ['generate']
INFO 05-30 08:31:09 [__init__.py:36] No IOProcessor plugins requested by the model


In [4]:
# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/finance_news_summarizer", split="train")

# 2. system_message 정의
# 데이터셋에 이미 포함된 system_prompt 열을 사용할 것이므로 따로 정의하지 않음

# 3. 원본 데이터의 type별 분포 출력
# 데이터셋에 type 열이 없으므로 전체 데이터 크기만 출력
print("전체 데이터 크기:", len(dataset))

# 4. train/test 분할 비율 설정 (0.5면 5:5로 분할)
test_ratio = 0.5

train_data = []
test_data = []

# 5. 전체 데이터의 인덱스를 train/test로 분할
data_indices = list(range(len(dataset)))
test_size = int(len(data_indices) * test_ratio)

test_data = data_indices[:test_size]
train_data = data_indices[test_size:]

# 6. OpenAI format으로 데이터 변환을 위한 함수
def format_data(sample):
    return {
        "messages": [
            {
                "role": "system",
                "content": sample["system_prompt"],
            },
            {
                "role": "user",
                "content": sample["user_prompt"],
            },
            {
                "role": "assistant",
                "content": str(sample["assistant"])
            },
        ],
    }

# 7. 분할된 데이터를 OpenAI format으로 변환
train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

# 8. 최종 데이터셋 크기 출력
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

전체 데이터 크기: 991

전체 데이터 분할 결과: Train 496개, Test 495개


In [5]:
def remove_think_blocks(text):
    if text is None:
        return ""

    text = str(text)
    # think 블록 제거
    text = text.replace("<think>\n\n</think>\n\n", "")

    return text.strip()

In [6]:
tokenizer = AutoTokenizer.from_pretrained('qwen3-4b-finance-new-summarizer/merged')

prompt_lst = []
label_lst = []

for row in test_dataset:
    prompt = row["messages"]

    text = tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=False
    )

    input_text = (
        text.split("<|im_start|>assistant")[0]
        + "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

    label = text.split("<think>\n\n</think>\n\n", 1)[1]

    prompt_lst.append(input_text)
    label_lst.append(label)

The tokenizer you are loading from 'qwen3-4b-finance-new-summarizer/merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [7]:
print(prompt_lst[0])

<|im_start|>system
당신은 주어진 뉴스로부터 종목에 영향을 주는 뉴스인지 판별하는 금융 뉴스 판별기입니다.
두 가지 답변 케이스가 존재하며 무조건 파이썬의 dictionary 형식으로 작성하십시오.
큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다. 따라서 주의하십시오.
아래 dictionary에서 각 value는 지시사항에 해당합니다. 지사사항을 따라 적지마십시오. 해당 지시사항에 따라 적절한 value를 채워넣으십시오.
해당사항이 없다면 빈 문자열 또는 빈 리스트로 적어야 합니다. 임의로 '없음' 등을 적어서는 안 됩니다.

만약 해당 뉴스가 특정 종목(회사)이 언급되지 않거나, 특정 종목(회사)와 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": False,
"summary": "여기에는 해당 뉴스를 요약해서 요약문을 작성하십시오"}

만약 해당 뉴스가 특정 종목(회사)들과 연관되었거나, 특정 종목(회사)과 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": True,
"positive_impact_stocks": ["파이썬 문자열 리스트의 형태로 이 뉴스가 긍정적인 영향을 줄것으로 추정되는 종목들의 이름을 작성하십시오. 약자로 적거나 별명으로 적지마십시오. 종목명으로 추정되는 한글명을 적으십시오. 뉴스로부터 추정할 수 있는 정확한 풀네임으로 적으십시오. 만약, 존재하지 않는다면 빈 리스트로 작성하십시오."],
"reason_for_positive_impact": "위의 종목들이 해당 뉴스로부터 긍정적인 영향을 받을 것으로 추정한 이유를 여기에다가 작성하십시오",
"positive_keywords": ["긍정적인 영향을 줄 것으로 추정되는 종목들이 존재했다면 여기에 긍정적인 영향을 주는데 근거가 되었던 주요한 명사 키워드들을 파이썬 문자열 리스트 형태로 작성

In [8]:
print(label_lst[0])

{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['현대상선', '대한통운', '한진', '삼성전자', 'LG전자'], 'positive_keywords': ['무역금융', '수출 지원', '임시선박', '물류비 지원', '첨단 산업 육성', '반도체'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '정부의 수출 지원 확대와 무역금융 규모 증가가 물류 및 전자 관련 기업들의 수출 및 운영에 긍정적인 영향을 미칠 것으로 예상되기 때문이다.', 'summary': '한국 정부가 수출 확대를 위해 무역금융을 40조 원 이상 확대하고 수출 중소기업의 물류비를 지원하기로 했습니다. 이는 수출 중심의 한국 경제 회복을 위한 대책이며, 반도체와 같은 첨단 산업 육성 전략도 포함됩니다.'}<|im_end|>



In [9]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
    stop=["<|im_end|>"],
)

In [10]:
preds = llm.generate(prompt_lst, sampling_params)
preds = [pred.outputs[0].text for pred in preds]

Adding requests:   0%|          | 0/495 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/495 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [11]:
print(preds[0])
print('---')
print(label_lst[0])

{"is_stock_related": True,
"positive_impact_stocks": ["반도체 업체", "첨단 산업 기업", "수출 기업"],
"reason_for_positive_impact": "정부가 수출 확대를 위해 무역금융 규모를 확대하고, 중소기업을 위한 물류비 지원 및 임시선박 투입, 해외 전시회 참가 지원 등 수출 기회를 늘리는 정책을 발표했기 때문에 관련 산업 및 수출 기업에 긍정적인 영향을 줄 것으로 예상됩니다.",
"positive_keywords": ["무역금융 확대", "물류비 지원", "임시선박 투입", "해외 전시회 참가 지원", "수출 기회 증가", "첨단 산업 육성"],
"negative_impact_stocks": [],
"reason_for_negative_impact": "",
"negative_keywords": [],
"summary": "정부는 올해 하반기 수출 확대를 위해 무역금융 규모를 40조 원 이상 확대하고, 중소기업의 물류난 해소를 위해 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 또한 수출 기회를 늘리기 위해 2500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하고, 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침할 계획입니다."}
---
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['현대상선', '대한통운', '한진', '삼성전자', 'LG전자'], 'positive_keywords': ['무역금융', '수출 지원', '임시선박', '물류비 지원', '첨단 산업 육성', '반도체'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '정부의 수출 지원 확대와 무역금융 규모 증가가 물류 및 전자 관련 기업들의 수출 및 운영에 긍정적인 영향을 미칠 것으

In [12]:
print(preds[25])
print('---')
print(label_lst[25])

{"is_stock_related": True,
"positive_impact_stocks": ["에너지 수출 기업", "수출 산업"], 
"reason_for_positive_impact": "상반기 수출 증가로 인해 에너지 수출 기업과 수출 산업이 긍정적인 영향을 받을 것으로 예상됩니다.",
"positive_keywords": ["수출", "에너지 수출", "수출 증가"],
"negative_impact_stocks": ["에너지 수입 기업", "에너지 관련 산업"],
"reason_for_negative_impact": "에너지 원자재 가격 급등으로 인한 수입액 증가로 인해 에너지 수입 기업과 에너지 관련 산업이 부정적인 영향을 받을 것으로 예상됩니다.",
"negative_keywords": ["에너지 수입", "에너지 원자재 가격", "수입액 증가"],
"summary": "한국의 올해 상반기 무역적자가 103억 달러로 역대 최대 규모를 기록했습니다. 수출은 15.6% 증가했으나, 에너지 원자재 가격 급등으로 인한 수입액 증가로 무역적자가 발생했습니다."}
---
{'is_stock_related': False, 'negative_impact_stocks': None, 'negative_keywords': None, 'positive_impact_stocks': None, 'positive_keywords': None, 'reason_for_negative_impact': None, 'reason_for_positive_impact': None, 'summary': '우리나라의 올해 상반기 무역적자가 103억 달러로 역대 최대를 기록했습니다. 수출은 15.6% 증가했으나, 에너지 원자재 가격 급등으로 수입액이 26.2% 증가하여 무역수지 적자가 발생했습니다.'}<|im_end|>



In [13]:
df_result = pd.DataFrame({
    "pred": preds,
    "label": label_lst,
})

df_result.to_pickle("base_pred_label.pkl")

In [14]:
import ast
import json
import re

import pandas as pd
from mecab import MeCab
from rouge import Rouge


def evaluate_one(pred, label):
    # ------------------------------------------------------------
    # 0. 기본 객체 준비
    # ------------------------------------------------------------
    # 한국어 ROUGE 계산을 위해 MeCab으로 형태소 단위 토큰화를 수행합니다.
    mecab = MeCab()
    rouge = Rouge()

    # 평가 대상 필드입니다.
    keys = [
        "is_stock_related",
        "negative_impact_stocks",
        "negative_keywords",
        "positive_impact_stocks",
        "positive_keywords",
        "reason_for_negative_impact",
        "reason_for_positive_impact",
        "summary",
    ]

    # ------------------------------------------------------------
    # 1. pred / label 파싱
    # ------------------------------------------------------------
    # dict, JSON 문자열, Python dict 문자열, ```json 코드블록``` 형태를 모두 처리합니다.
    # 그래도 파싱이 안 되면 해당 샘플은 parse_failed로 처리합니다.
    def parse_record(x):
        if isinstance(x, dict):
            return x, False

        text = str(x).strip()

        # 코드블록 제거
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

        # 가장 바깥쪽 { ... }만 추출
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1 or end <= start:
            return {}, True

        dict_text = text[start:end + 1]
        inner_text = text[start + 1:end]

        # 1순위: 정상 JSON 파싱
        try:
            parsed = json.loads(dict_text)
            if isinstance(parsed, dict):
                for key in keys:
                    parsed.setdefault(key, None)
                return parsed, False
        except Exception:
            pass

        # 2순위: Python dict 문자열 파싱
        try:
            parsed = ast.literal_eval(dict_text)
            if isinstance(parsed, dict):
                for key in keys:
                    parsed.setdefault(key, None)
                return parsed, False
        except Exception:
            pass

        # 3순위: key 위치 기준 수동 파싱
        # 예: summary 안에 '똘똘한 한 채'처럼 작은따옴표가 들어가 ast 파싱이 깨지는 경우 대응
        parsed = {}
        key_matches = []

        for key in keys:
            pattern = rf"""(['"]){re.escape(key)}\1\s*:"""
            match = re.search(pattern, inner_text)

            if match:
                key_matches.append((match.start(), match.end(), key))

        key_matches = sorted(key_matches, key=lambda x: x[0])

        if not key_matches:
            return {}, True

        for idx, (_, value_start, key) in enumerate(key_matches):
            if idx + 1 < len(key_matches):
                next_key_start = key_matches[idx + 1][0]
                value_raw = inner_text[value_start:next_key_start].strip()
            else:
                value_raw = inner_text[value_start:].strip()

            value_raw = value_raw.rstrip().rstrip(",")

            if value_raw in ["None", "null"]:
                parsed[key] = None

            elif value_raw in ["True", "true"]:
                parsed[key] = True

            elif value_raw in ["False", "false"]:
                parsed[key] = False

            elif value_raw.startswith("[") and value_raw.endswith("]"):
                try:
                    parsed[key] = json.loads(value_raw)
                except Exception:
                    try:
                        parsed[key] = ast.literal_eval(value_raw)
                    except Exception:
                        parsed[key] = []

            else:
                value_raw = value_raw.strip()

                # 문자열 바깥쪽 따옴표만 제거하고 내부 따옴표는 유지합니다.
                if len(value_raw) >= 2 and value_raw[0] == '"' and value_raw[-1] == '"':
                    value_raw = value_raw[1:-1]
                elif len(value_raw) >= 2 and value_raw[0] == "'" and value_raw[-1] == "'":
                    value_raw = value_raw[1:-1]

                parsed[key] = value_raw

        for key in keys:
            parsed.setdefault(key, None)

        return parsed, False

    pred, pred_parse_failed = parse_record(pred)
    label, label_parse_failed = parse_record(label)

    result = {}

    # ------------------------------------------------------------
    # 2. 파싱 실패 처리
    # ------------------------------------------------------------
    # pred 또는 label 중 하나라도 파싱 실패하면 평가 불가능 샘플로 보고 0점 처리합니다.
    if pred_parse_failed or label_parse_failed:
        result["parse_failed"] = 1.0
        result["pred_parse_failed"] = 1.0 if pred_parse_failed else 0.0
        result["label_parse_failed"] = 1.0 if label_parse_failed else 0.0
        result["is_stock_related_acc"] = 0.0

        list_fields = [
            "negative_impact_stocks",
            "positive_impact_stocks",
            "negative_keywords",
            "positive_keywords",
        ]

        for field in list_fields:
            result[f"{field}_precision"] = 0.0
            result[f"{field}_recall"] = 0.0
            result[f"{field}_f1"] = 0.0
            result[f"{field}_exact_match"] = 0.0

        result["list_macro_f1"] = 0.0

        text_fields = [
            "reason_for_negative_impact",
            "reason_for_positive_impact",
            "summary",
        ]

        for field in text_fields:
            result[f"{field}_rouge1_precision"] = 0.0
            result[f"{field}_rouge1_recall"] = 0.0
            result[f"{field}_rouge1_f1"] = 0.0

            result[f"{field}_rouge2_precision"] = 0.0
            result[f"{field}_rouge2_recall"] = 0.0
            result[f"{field}_rouge2_f1"] = 0.0

            result[f"{field}_rougeL_precision"] = 0.0
            result[f"{field}_rougeL_recall"] = 0.0
            result[f"{field}_rougeL_f1"] = 0.0

        result["rouge_macro_f1"] = 0.0

        return result

    result["parse_failed"] = 0.0
    result["pred_parse_failed"] = 0.0
    result["label_parse_failed"] = 0.0

    # ------------------------------------------------------------
    # 3. is_stock_related 평가
    # ------------------------------------------------------------
    # True/False 분류 정확도입니다.
    pred_is_stock = pred.get("is_stock_related")
    label_is_stock = label.get("is_stock_related")

    result["is_stock_related_acc"] = 1.0 if pred_is_stock == label_is_stock else 0.0

    # ------------------------------------------------------------
    # 4. 오분류 반영 기준
    # ------------------------------------------------------------
    # 둘 다 False일 때만 종목/키워드/reason 평가를 제외합니다.
    # 하나라도 True이면 해당 필드들을 평가하여 오분류 벌점이 반영되게 합니다.
    need_stock_fields_eval = (pred_is_stock is True) or (label_is_stock is True)

    # ------------------------------------------------------------
    # 5. 리스트 평가 함수
    # ------------------------------------------------------------
    def normalize_item(x):
        # 공백 제거 + 소문자 변환
        # 예: "첨단 산업" -> "첨단산업"
        return re.sub(r"\s+", "", str(x).lower()).strip()

    def to_clean_list(values):
        # None, 문자열, 기타 타입을 모두 리스트로 정리합니다.
        if values is None:
            return []

        if isinstance(values, str):
            values = [values]

        if not isinstance(values, list):
            values = [values]

        return [
            normalize_item(x)
            for x in values
            if str(x).strip()
        ]

    def exact_list_f1(pred_values, label_values):
        # 종목명처럼 정확히 일치해야 하는 필드에 사용합니다.
        pred_items = to_clean_list(pred_values)
        label_items = to_clean_list(label_values)

        pred_set = set(pred_items)
        label_set = set(label_items)

        if len(pred_set) == 0 and len(label_set) == 0:
            return 1.0, 1.0, 1.0, 1.0

        if len(pred_set) == 0 or len(label_set) == 0:
            return 0.0, 0.0, 0.0, 0.0

        tp = len(pred_set & label_set)

        precision = tp / len(pred_set)
        recall = tp / len(label_set)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        exact_match = 1.0 if pred_set == label_set else 0.0

        return precision, recall, f1, exact_match

    def soft_keyword_f1(pred_values, label_values):
        # 키워드는 exact match + 포함 관계를 허용합니다.
        # 예: "첨단 산업"과 "첨단 산업 육성"은 매칭
        pred_items = to_clean_list(pred_values)
        label_items = to_clean_list(label_values)

        if len(pred_items) == 0 and len(label_items) == 0:
            return 1.0, 1.0, 1.0, 1.0

        if len(pred_items) == 0 or len(label_items) == 0:
            return 0.0, 0.0, 0.0, 0.0

        matched_label_idx = set()
        tp = 0

        for pred_item in pred_items:
            for idx, label_item in enumerate(label_items):
                if idx in matched_label_idx:
                    continue

                if pred_item == label_item:
                    matched_label_idx.add(idx)
                    tp += 1
                    break

                if pred_item in label_item or label_item in pred_item:
                    matched_label_idx.add(idx)
                    tp += 1
                    break

        precision = tp / len(pred_items)
        recall = tp / len(label_items)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        exact_match = 1.0 if set(pred_items) == set(label_items) else 0.0

        return precision, recall, f1, exact_match

    # ------------------------------------------------------------
    # 6. 종목/키워드 리스트 평가
    # ------------------------------------------------------------
    # 종목 필드는 exact match, 키워드 필드는 soft match로 평가합니다.
    list_fields = [
        "negative_impact_stocks",
        "positive_impact_stocks",
        "negative_keywords",
        "positive_keywords",
    ]

    stock_fields = [
        "negative_impact_stocks",
        "positive_impact_stocks",
    ]

    keyword_fields = [
        "negative_keywords",
        "positive_keywords",
    ]

    list_f1s = []

    if need_stock_fields_eval:
        for field in list_fields:
            pred_values = pred.get(field) or []
            label_values = label.get(field) or []

            if field in stock_fields:
                precision, recall, f1, exact_match = exact_list_f1(
                    pred_values,
                    label_values,
                )

            elif field in keyword_fields:
                precision, recall, f1, exact_match = soft_keyword_f1(
                    pred_values,
                    label_values,
                )

            else:
                precision, recall, f1, exact_match = exact_list_f1(
                    pred_values,
                    label_values,
                )

            result[f"{field}_precision"] = precision
            result[f"{field}_recall"] = recall
            result[f"{field}_f1"] = f1
            result[f"{field}_exact_match"] = exact_match

            list_f1s.append(f1)

        # 4개 리스트 필드 F1 평균입니다.
        result["list_macro_f1"] = sum(list_f1s) / len(list_f1s)

    else:
        # label=False, pred=False이면 종목/키워드는 평가 대상이 아닙니다.
        for field in list_fields:
            result[f"{field}_precision"] = None
            result[f"{field}_recall"] = None
            result[f"{field}_f1"] = None
            result[f"{field}_exact_match"] = None

        result["list_macro_f1"] = None

    # ------------------------------------------------------------
    # 7. 텍스트 필드 ROUGE 평가
    # ------------------------------------------------------------
    # 둘 다 False이면 summary만 평가합니다.
    # 하나라도 True이면 reason 2개와 summary를 모두 평가합니다.
    if need_stock_fields_eval:
        text_fields = [
            "reason_for_negative_impact",
            "reason_for_positive_impact",
            "summary",
        ]
    else:
        text_fields = [
            "summary",
        ]

    rouge_f1s = []

    for field in text_fields:
        pred_text = "" if pred.get(field) is None else str(pred.get(field)).strip()
        label_text = "" if label.get(field) is None else str(label.get(field)).strip()

        if pred_text == "" and label_text == "":
            r1_p, r1_r, r1_f = 1.0, 1.0, 1.0
            r2_p, r2_r, r2_f = 1.0, 1.0, 1.0
            rl_p, rl_r, rl_f = 1.0, 1.0, 1.0

        elif pred_text == "" or label_text == "":
            r1_p, r1_r, r1_f = 0.0, 0.0, 0.0
            r2_p, r2_r, r2_f = 0.0, 0.0, 0.0
            rl_p, rl_r, rl_f = 0.0, 0.0, 0.0

        else:
            pred_tokenized = " ".join(mecab.morphs(pred_text))
            label_tokenized = " ".join(mecab.morphs(label_text))

            # hypothesis=pred, reference=label
            scores = rouge.get_scores(pred_tokenized, label_tokenized)[0]

            r1_p = scores["rouge-1"]["p"]
            r1_r = scores["rouge-1"]["r"]
            r1_f = scores["rouge-1"]["f"]

            r2_p = scores["rouge-2"]["p"]
            r2_r = scores["rouge-2"]["r"]
            r2_f = scores["rouge-2"]["f"]

            rl_p = scores["rouge-l"]["p"]
            rl_r = scores["rouge-l"]["r"]
            rl_f = scores["rouge-l"]["f"]

        result[f"{field}_rouge1_precision"] = r1_p
        result[f"{field}_rouge1_recall"] = r1_r
        result[f"{field}_rouge1_f1"] = r1_f

        result[f"{field}_rouge2_precision"] = r2_p
        result[f"{field}_rouge2_recall"] = r2_r
        result[f"{field}_rouge2_f1"] = r2_f

        result[f"{field}_rougeL_precision"] = rl_p
        result[f"{field}_rougeL_recall"] = rl_r
        result[f"{field}_rougeL_f1"] = rl_f

        rouge_f1s.extend([r1_f, r2_f, rl_f])

    # 텍스트 필드 전체 ROUGE F1 평균입니다.
    result["rouge_macro_f1"] = sum(rouge_f1s) / len(rouge_f1s)

    return result


def evaluate_all(preds, labels):
    # preds[i]와 labels[i]를 한 쌍으로 평가합니다.
    rows = [
        evaluate_one(pred, label)
        for pred, label in zip(preds, labels)
    ]

    # 샘플별 평가 결과를 DataFrame으로 정리합니다.
    df = pd.DataFrame(rows)

    # 전체 평균 지표를 계산합니다.
    # None 값은 pandas에서 자동으로 평균 계산에서 제외됩니다.
    avg = df.mean(numeric_only=True).to_dict()

    return df, avg

In [15]:
df_eval, avg_eval = evaluate_all(preds, label_lst)

display(df_eval)

,parse_failed,pred_parse_failed,label_parse_failed,is_stock_related_acc,negative_impact_stocks_precision,negative_impact_stocks_recall,negative_impact_stocks_f1,negative_impact_stocks_exact_match,positive_impact_stocks_precision,positive_impact_stocks_recall,...,summary_rouge1_precision,summary_rouge1_recall,summary_rouge1_f1,summary_rouge2_precision,summary_rouge2_recall,summary_rouge2_f1,summary_rougeL_precision,summary_rougeL_recall,summary_rougeL_f1,rouge_macro_f1
0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.000000,0.0,...,0.467742,0.659091,0.547170,0.282051,0.423077,0.338462,0.451613,0.636364,0.528302,0.622020
1,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.623188,0.728814,0.671875,0.404762,0.465753,0.433121,0.608696,0.711864,0.656250,0.587082
2,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.166667,1.0,...,0.462687,0.720930,0.563636,0.258427,0.469388,0.333333,0.432836,0.674419,0.527273,0.629474
3,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,...,0.400000,0.722222,0.514851,0.207317,0.404762,0.274194,0.338462,0.611111,0.435644,0.581007
4,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,...,0.728814,0.544304,0.623188,0.424242,0.271845,0.331361,0.661017,0.493671,0.565217,0.168863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,...,0.363636,0.606061,0.454545,0.156250,0.256410,0.194175,0.327273,0.545455,0.409091,0.586293
491,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.866667,0.838710,0.852459,0.750000,0.705882,0.727273,0.866667,0.838710,0.852459,0.810730
492,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
493,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0,...,0.578125,0.578125,0.578125,0.292683,0.300000,0.296296,0.500000,0.500000,0.500000,0.556564


In [16]:
metric_desc = {
    "parse_failed": "pred 또는 label 파싱 실패 비율",
    "pred_parse_failed": "예측값 파싱 실패 비율",
    "label_parse_failed": "레이블 파싱 실패 비율",

    "is_stock_related_acc": "주식 관련 여부 True/False 분류 정확도",

    "negative_impact_stocks_f1": "부정 영향 종목 리스트 추출 F1",
    "positive_impact_stocks_f1": "긍정 영향 종목 리스트 추출 F1",
    "negative_keywords_f1": "부정 키워드 리스트 추출 F1",
    "positive_keywords_f1": "긍정 키워드 리스트 추출 F1",
    "list_macro_f1": "위 4개 리스트형 필드 F1의 평균",

    "reason_for_negative_impact_rougeL_f1": "부정 영향 이유 문장의 MeCab 기반 ROUGE-L F1",
    "reason_for_positive_impact_rougeL_f1": "긍정 영향 이유 문장의 MeCab 기반 ROUGE-L F1",
    "summary_rougeL_f1": "요약문 MeCab 기반 ROUGE-L F1",

    "rouge_macro_f1": "텍스트 필드 ROUGE F1 전체 평균",
}

main_metrics = [
    "parse_failed",
    "pred_parse_failed",
    "label_parse_failed",

    "is_stock_related_acc",

    "negative_impact_stocks_f1",
    "positive_impact_stocks_f1",
    "negative_keywords_f1",
    "positive_keywords_f1",
    "list_macro_f1",

    "reason_for_negative_impact_rougeL_f1",
    "reason_for_positive_impact_rougeL_f1",
    "summary_rougeL_f1",

    "rouge_macro_f1",
]

avg_eval_main_df = pd.DataFrame(
    [
        {
            "metric": k,
            "description": metric_desc.get(k, ""),
            "score": avg_eval.get(k),
        }
        for k in main_metrics
    ]
)

display(avg_eval_main_df.style.format({"score": "{:.4f}"}))

,metric,description,score
0,parse_failed,pred 또는 label 파싱 실패 비율,0.0141
1,pred_parse_failed,예측값 파싱 실패 비율,0.0141
2,label_parse_failed,레이블 파싱 실패 비율,0.0000
3,is_stock_related_acc,주식 관련 여부 True/False 분류 정확도,0.8040
4,negative_impact_stocks_f1,부정 영향 종목 리스트 추출 F1,0.7846
5,positive_impact_stocks_f1,긍정 영향 종목 리스트 추출 F1,0.6280
6,negative_keywords_f1,부정 키워드 리스트 추출 F1,0.7528
7,positive_keywords_f1,긍정 키워드 리스트 추출 F1,0.5574
8,list_macro_f1,위 4개 리스트형 필드 F1의 평균,0.6807
9,reason_for_negative_impact_rougeL_f1,부정 영향 이유 문장의 MeCab 기반 ROUGE-L F1,0.7482
